# Forecasting

**Autor:** Cleber de Jesus Salustiano  
**Matricula:** 20251mpca0170

**Objetivo:** usar as tecnicas de forecasting apresentadas na aula em uma base de dados diferente da apresentada em aula, adotando uma baseline **Naive** e comparando os resultados obtidos para a previsao da temperatura minima do dia seguinte.

## Base utilizada

Foi usada a base **Daily Minimum Temperatures in Melbourne (1981-1990)**.

Em poucas palavras: e uma serie temporal **univariada** com a temperatura minima diaria registrada em Melbourne, Australia, ao longo de 10 anos.

Fonte do CSV: `https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv`

In [1]:
import importlib
import subprocess
import sys

required_packages = ['numpy', 'pandas', 'matplotlib', 'tensorflow']

for package in required_packages:
    try:
        importlib.import_module(package)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('Dependencias verificadas com sucesso.')


Matplotlib is building the font cache; this may take a moment.


Dependencias verificadas com sucesso.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

plt.style.use('seaborn-v0_8-whitegrid')
keras.utils.set_random_seed(42)


## Configuracao do experimento

Os principais parametros usados para gerar o dataset a partir do array ficam visiveis nesta secao.

In [3]:
DATA_URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv'

sequence_length = 14
sampling_rate = 1
delay = 1
batch_size = 32
epochs = 20

train_ratio = 0.70
val_ratio = 0.15
test_ratio = 0.15

assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9

target_offset = sequence_length + delay - 1

print(f'sequence_length = {sequence_length}')
print(f'sampling_rate = {sampling_rate}')
print(f'delay = {delay}')
print(f'batch_size = {batch_size}')
print(f'target_offset = {target_offset}')


sequence_length = 14
sampling_rate = 1
delay = 1
batch_size = 32
target_offset = 14


## Justificativa dos parametros

- `sequence_length = 14`: usa duas semanas de historico, o que ajuda a capturar o comportamento recente da serie sem criar janelas muito grandes.
- `sampling_rate = 1`: como a serie ja e diaria, cada observacao consecutiva deve ser aproveitada.
- `delay = 1`: o alvo e a temperatura minima do dia seguinte, caracterizando uma previsao de um passo a frente.
- `batch_size = 32`: valor simples e estavel para treinar os modelos com essa base.

**Observacao importante:** ao usar `timeseries_dataset_from_array`, o alvo precisa ser deslocado em `sequence_length + delay - 1`.
Dessa forma, cada janela com 14 valores do array fica alinhada com a temperatura do dia seguinte.

## Leitura da base

Nesta atividade, a previsao sera feita sobre uma serie temporal univariada, usando apenas a coluna `Temp`.

In [4]:
df = pd.read_csv(DATA_URL)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

df.head()


,Date,Temp
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


In [5]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 3650 entries, 0 to 3649
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    3650 non-null   datetime64[us]
 1   Temp    3650 non-null   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 57.2 KB


In [6]:
temperatures = df['Temp'].to_numpy(dtype='float32')
dates = df['Date']

print(f'Total de observacoes: {len(temperatures)}')
print(f'Periodo: {dates.min().date()} ate {dates.max().date()}')
print(f'Temperatura minima observada: {temperatures.min():.1f} C')
print(f'Temperatura maxima observada: {temperatures.max():.1f} C')


Total de observacoes: 3650
Periodo: 1981-01-01 ate 1990-12-31
Temperatura minima observada: 0.0 C
Temperatura maxima observada: 26.3 C


## Visualizacao inicial da serie

In [7]:
plt.figure(figsize=(14, 4))
plt.plot(dates, temperatures, color='tab:blue', linewidth=1)
plt.title('Temperatura minima diaria em Melbourne (1981-1990)')
plt.xlabel('Data')
plt.ylabel('Temperatura minima (C)')
plt.show()


/var/folders/gt/zkqdq7pj73nblw9yvwlq1_3c0000gn/T/ipykernel_56702/886669253.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
plt.figure(figsize=(14, 4))
plt.plot(dates.iloc[:60], temperatures[:60], marker='o', linewidth=1.5, color='tab:orange')
plt.title('Primeiros 60 dias da serie')
plt.xlabel('Data')
plt.ylabel('Temperatura minima (C)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


/var/folders/gt/zkqdq7pj73nblw9yvwlq1_3c0000gn/T/ipykernel_56702/3896818046.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Separacao treino, validacao e teste

A divisao e cronologica, sem embaralhar a serie, para respeitar a ordem temporal dos dados.

In [9]:
num_samples = len(temperatures)
train_end = int(num_samples * train_ratio)
val_end = int(num_samples * (train_ratio + val_ratio))

print(f'Treino: 0 ate {train_end - 1} ({train_end} amostras)')
print(f'Validacao: {train_end} ate {val_end - 1} ({val_end - train_end} amostras)')
print(f'Teste: {val_end} ate {num_samples - 1} ({num_samples - val_end} amostras)')


Treino: 0 ate 2554 (2555 amostras)
Validacao: 2555 ate 3101 (547 amostras)
Teste: 3102 ate 3649 (548 amostras)


## Normalizacao

A normalizacao usa apenas a media e o desvio padrao do conjunto de treino, evitando vazamento de informacao.

In [10]:
train_mean = temperatures[:train_end].mean()
train_std = temperatures[:train_end].std()

normalized_temperatures = (temperatures - train_mean) / train_std

print(f'Media do treino: {train_mean:.3f}')
print(f'Desvio padrao do treino: {train_std:.3f}')


Media do treino: 10.982
Desvio padrao do treino: 4.088


## Geracao dos datasets com `timeseries_dataset_from_array`

Cada entrada tera 14 dias consecutivos de temperatura normalizada, e cada alvo sera a temperatura minima do dia seguinte em escala original.

In [11]:
def make_dataset(start_index, end_index, shuffle=False):
    inputs = normalized_temperatures[start_index:end_index - delay]
    targets = temperatures[start_index + target_offset:end_index]

    return keras.utils.timeseries_dataset_from_array(
        data=inputs[..., np.newaxis],
        targets=targets,
        sequence_length=sequence_length,
        sampling_rate=sampling_rate,
        batch_size=batch_size,
        shuffle=shuffle,
    )

train_dataset = make_dataset(0, train_end, shuffle=True)
val_dataset = make_dataset(train_end, val_end)
test_dataset = make_dataset(val_end, num_samples)


In [12]:
for samples, targets in train_dataset.take(1):
    print('samples shape:', samples.shape)
    print('targets shape:', targets.shape)
    break


samples shape: (32, 14, 1)
targets shape: (32,)


In [13]:
for sample_batch, target_batch in train_dataset.take(1):
    sample_window = sample_batch[0].numpy().squeeze()
    target_value = float(target_batch[0].numpy())
    denormalized_window = sample_window * train_std + train_mean

    print('Janela de entrada (14 dias, escala original):')
    print(np.round(denormalized_window, 2))
    print(f'Alvo correspondente (dia seguinte): {target_value:.2f} C')

    plt.figure(figsize=(10, 4))
    plt.plot(range(1, sequence_length + 1), denormalized_window, marker='o', label='Janela de entrada')
    plt.scatter(sequence_length + delay, target_value, color='red', s=80, label='Alvo (dia seguinte)')
    plt.xticks(range(1, sequence_length + delay + 1))
    plt.xlabel('Posicao temporal dentro da amostra')
    plt.ylabel('Temperatura minima (C)')
    plt.title('Exemplo de janela e alvo com delay=1')
    plt.legend()
    plt.show()
    break


Janela de entrada (14 dias, escala original):
[13.9  7.7  9.5  7.6  6.9  6.8  5.8  6.   8.3  9.1 12.5 13.2 16.2 12.5]
Alvo correspondente (dia seguinte): 11.80 C


/var/folders/gt/zkqdq7pj73nblw9yvwlq1_3c0000gn/T/ipykernel_56702/4221895051.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Baseline Naive

A baseline Naive assume que a temperatura do proximo dia sera igual a ultima temperatura observada na janela.

In [14]:
def evaluate_naive_method(dataset):
    total_abs_err = 0.0
    samples_seen = 0

    for samples, targets in dataset:
        preds = samples[:, -1, 0].numpy() * train_std + train_mean
        total_abs_err += np.sum(np.abs(preds - targets.numpy()))
        samples_seen += len(targets)

    return total_abs_err / samples_seen

naive_val_mae = evaluate_naive_method(val_dataset)
naive_test_mae = evaluate_naive_method(test_dataset)

print(f'Naive MAE (validacao): {naive_val_mae:.4f}')
print(f'Naive MAE (teste): {naive_test_mae:.4f}')


Naive MAE (validacao): 2.0015
Naive MAE (teste): 2.0453


## Funcoes auxiliares para treinamento e avaliacao

In [15]:
def build_and_compile(model):
    model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])
    return model

def fit_and_evaluate(model, model_name):
    early_stopping = EarlyStopping(patience=5, restore_best_weights=True)

    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=epochs,
        callbacks=[early_stopping],
        verbose=1,
    )

    test_loss, test_mae = model.evaluate(test_dataset, verbose=0)
    best_val_mae = min(history.history['val_mae'])

    print(f'{model_name} - melhor val_mae: {best_val_mae:.4f}')
    print(f'{model_name} - test_mae: {test_mae:.4f}')

    return history, best_val_mae, test_mae


## Modelo denso

Arquitetura: `Flatten -> Dense(16, relu) -> Dense(1)`

In [16]:
dense_model = build_and_compile(
    keras.Sequential([
        keras.Input(shape=(sequence_length, 1)),
        layers.Flatten(),
        layers.Dense(16, activation='relu'),
        layers.Dense(1),
    ])
)

dense_history, dense_val_mae, dense_test_mae = fit_and_evaluate(dense_model, 'Dense')


Epoch 1/20



 1/80 ━━━━━━━━━━━━━━━━━━━━ 18s 229ms/step - loss: 140.6607 - mae: 11.1707


80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 117.2670 - mae: 9.9207 - val_loss: 124.9554 - val_mae: 10.4321


Epoch 2/20



 1/80 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 85.5346 - mae: 8.0400

...


80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 881us/step - loss: 6.8943 - mae: 2.0641 - val_loss: 6.1478 - val_mae: 1.9704


Epoch 20/20



 1/80 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 7.1485 - mae: 2.2133


80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 855us/step - loss: 6.8051 - mae: 2.0497 - val_loss: 6.0767 - val_mae: 1.9627


Dense - melhor val_mae: 1.9627
Dense - test_mae: 1.8406


## Modelo convolucional 1D

Arquitetura: `Conv1D(16, 3, relu) -> MaxPooling1D(2) -> Conv1D(16, 3, relu) -> GlobalAveragePooling1D -> Dense(1)`

In [17]:
conv_model = build_and_compile(
    keras.Sequential([
        keras.Input(shape=(sequence_length, 1)),
        layers.Conv1D(16, 3, activation='relu'),
        layers.MaxPooling1D(2),
        layers.Conv1D(16, 3, activation='relu'),
        layers.GlobalAveragePooling1D(),
        layers.Dense(1),
    ])
)

conv_history, conv_val_mae, conv_test_mae = fit_and_evaluate(conv_model, 'Conv1D')


Epoch 1/20



 1/80 ━━━━━━━━━━━━━━━━━━━━ 20s 265ms/step - loss: 137.9309 - mae: 11.3215


80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 94.4382 - mae: 8.8409 - val_loss: 64.6973 - val_mae: 7.3785


Epoch 2/20



 1/80 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 42.9180 - mae: 5.6729

...

Epoch 20/20



 1/80 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 8.7558 - mae: 2.2484


66/80 ━━━━━━━━━━━━━━━━━━━━ 0s 769us/step - loss: 8.6437 - mae: 2.3134


80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 8.0716 - mae: 2.2481 - val_loss: 7.4080 - val_mae: 2.1244


Conv1D - melhor val_mae: 2.1244
Conv1D - test_mae: 1.8910


## Modelo recorrente LSTM

Arquitetura: `LSTM(16) -> Dense(1)`

In [18]:
lstm_model = build_and_compile(
    keras.Sequential([
        keras.Input(shape=(sequence_length, 1)),
        layers.LSTM(16),
        layers.Dense(1),
    ])
)

lstm_history, lstm_val_mae, lstm_test_mae = fit_and_evaluate(lstm_model, 'LSTM')


Epoch 1/20



 1/80 ━━━━━━━━━━━━━━━━━━━━ 38s 483ms/step - loss: 125.0821 - mae: 10.5873


42/80 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 134.3963 - mae: 10.8213   


80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 113.3193 - mae: 9.8870 - val_loss: 91.2023 - val_mae: 8.8302


Epoch 2/20


...

Epoch 20/20



 1/80 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 6.9348 - mae: 2.0520


45/80 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 6.8495 - mae: 2.0370 


80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 6.3744 - mae: 1.9741 - val_loss: 5.7125 - val_mae: 1.8772


LSTM - melhor val_mae: 1.8772
LSTM - test_mae: 1.7568


## Comparacao das curvas de validacao

In [19]:
plt.figure(figsize=(10, 5))
plt.plot(dense_history.history['val_mae'], label='Dense')
plt.plot(conv_history.history['val_mae'], label='Conv1D')
plt.plot(lstm_history.history['val_mae'], label='LSTM')
plt.axhline(naive_val_mae, color='black', linestyle='--', label='Naive baseline')
plt.xlabel('Epoca')
plt.ylabel('MAE de validacao')
plt.title('Comparacao do desempenho na validacao')
plt.legend()
plt.show()


/var/folders/gt/zkqdq7pj73nblw9yvwlq1_3c0000gn/T/ipykernel_56702/1477836683.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Comparacao final dos resultados

In [20]:
results = pd.DataFrame([
    {'modelo': 'Naive', 'val_mae': naive_val_mae, 'test_mae': naive_test_mae},
    {'modelo': 'Dense', 'val_mae': dense_val_mae, 'test_mae': dense_test_mae},
    {'modelo': 'Conv1D', 'val_mae': conv_val_mae, 'test_mae': conv_test_mae},
    {'modelo': 'LSTM', 'val_mae': lstm_val_mae, 'test_mae': lstm_test_mae},
])

results = results.sort_values('test_mae').reset_index(drop=True)
results


,modelo,val_mae,test_mae
0,LSTM,1.877208,1.756802
1,Dense,1.962747,1.840648
2,Conv1D,2.124351,1.891011
3,Naive,2.001501,2.045319


In [21]:
plt.figure(figsize=(8, 4))
plt.bar(results['modelo'], results['test_mae'], color=['gray', 'tab:blue', 'tab:orange', 'tab:green'])
plt.ylabel('MAE no teste')
plt.title('Comparacao final entre baseline e redes neurais')
plt.show()


/var/folders/gt/zkqdq7pj73nblw9yvwlq1_3c0000gn/T/ipykernel_56702/2217131854.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Conclusao curta

In [22]:
best_neural = results[results['modelo'] != 'Naive'].sort_values('test_mae').iloc[0]

print('Resumo automatico da atividade:')
print('- Base usada: temperatura minima diaria de Melbourne entre 1981 e 1990.')
print('- Baseline: previsao Naive, repetindo o ultimo valor observado.')
print(f"- Melhor modelo neural no teste: {best_neural['modelo']} com MAE = {best_neural['test_mae']:.4f}.")
print(f'- Baseline Naive no teste: MAE = {naive_test_mae:.4f}.')

if float(best_neural['test_mae']) < float(naive_test_mae):
    print('- Pelo menos um modelo neural superou a baseline Naive no conjunto de teste.')
else:
    print('- Nenhum modelo neural superou a baseline Naive no conjunto de teste; isso deve ser registrado de forma honesta na conclusao.')


Resumo automatico da atividade:
- Base usada: temperatura minima diaria de Melbourne entre 1981 e 1990.
- Baseline: previsao Naive, repetindo o ultimo valor observado.
- Melhor modelo neural no teste: LSTM com MAE = 1.7568.
- Baseline Naive no teste: MAE = 2.0453.
- Pelo menos um modelo neural superou a baseline Naive no conjunto de teste.
